## India Pay by Role

NASSCOM/MOSPI 2023 data, shown in US dollars two ways: **market USD** (paycheck converted at 82.6 INR/USD) and **PPP USD** (local buying power, ÷22 INR/PPP USD).

In [1]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import plotly.express as px

ROLE_LABELS = {
    'software_engineer': 'Software Engineer', 'lawyer': 'Lawyer',
    'physician': 'Physician', 'financial_analyst': 'Financial Analyst',
    'registered_nurse': 'Registered Nurse', 'civil_engineer': 'Civil Engineer',
    'construction_laborer': 'Construction Laborer', 'farm_worker': 'Farm Worker',
    'manufacturing_worker': 'Manufacturing Worker', 'retail_worker': 'Retail Worker',
}

df = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_india_data.csv')
df['role_label'] = df['role'].map(ROLE_LABELS)

In [2]:
mid_order = (
    df[df['career_stage'] == 'mid']
    .sort_values('median_salary_usd', ascending=False)['role_label'].tolist()
)
stage_colors = {'entry': '#a1d99b', 'mid': '#31a354', 'senior': '#006d2c'}

fig = px.bar(
    df, x='role_label', y='median_salary_usd', color='career_stage',
    barmode='group',
    category_orders={'role_label': mid_order, 'career_stage': ['entry', 'mid', 'senior']},
    color_discrete_map=stage_colors,
    title='India Annual Salary by Role — Market USD (82.6 INR/USD, 2023)',
    labels={'role_label': 'Role', 'median_salary_usd': 'Annual Salary (market USD)', 'career_stage': 'Stage'},
)
fig.update_layout(xaxis_tickangle=-35)
fig.show()

In [3]:
fig2 = px.bar(
    df, x='role_label', y='median_salary_ppp_usd', color='career_stage',
    barmode='group',
    category_orders={'role_label': mid_order, 'career_stage': ['entry', 'mid', 'senior']},
    color_discrete_map=stage_colors,
    title='India Annual Salary by Role — PPP USD (local buying power, ÷22 INR/PPP USD)',
    labels={'role_label': 'Role', 'median_salary_ppp_usd': 'Annual Salary (PPP USD)', 'career_stage': 'Stage'},
)
fig2.update_layout(xaxis_tickangle=-35)
fig2.show()

In [4]:
mid_df = df[df['career_stage'] == 'mid'].copy()
swe_mid = mid_df[mid_df['role'] == 'software_engineer']['median_salary_usd'].values[0]
mid_df['swe_multiple'] = (swe_mid / mid_df['median_salary_usd']).round(2)
mid_df = mid_df[mid_df['role'] != 'software_engineer'].sort_values('swe_multiple', ascending=True)
mid_df['role_label'] = mid_df['role'].map(ROLE_LABELS)

fig3 = px.bar(
    mid_df, x='swe_multiple', y='role_label', orientation='h',
    color='swe_multiple', color_continuous_scale='Greens',
    title=f'India: SWE Mid-Career Pay Multiple over Peer Roles (2023)<br>'
          f'<sup>SWE median = ${swe_mid:,.0f} market USD</sup>',
    labels={'swe_multiple': 'SWE / Role Salary Multiple', 'role_label': 'Role'},
)
fig3.add_vline(x=1.0, line_dash='dot', line_color='red', annotation_text='Equal pay')
fig3.show()